# 5G 2.6 Clusteriing + Sequencing

In [5]:
import pandas as pd
import numpy as np
import glob
import os
import folium
import geopandas as gpd
from folium.features import RegularPolygonMarker
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors
from shapely.geometry import Point
from shapely.ops import voronoi_diagram, unary_union
from branca.element import Template, MacroElement

# 1. SETUP & DATA LOADING
base_folder = "5G 2.6 site list"
vip_folder = "VIP"
result_folder = os.path.join(base_folder, "Result")
border_folder = os.path.join(result_folder, "Border")

for folder in [result_folder, border_folder]:
    if not os.path.exists(folder):
        os.makedirs(folder)

def safe_read(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1')

csv_files = [f for f in glob.glob(os.path.join(base_folder, "*.csv")) if "Result" not in f and "VIP" not in f]
if not csv_files:
    raise FileNotFoundError("Could not find the site list CSV!")

df_main = safe_read(csv_files[0])
df_main.columns = df_main.columns.str.strip()
df_main['is_hvc'] = df_main['HVC'].astype(str).str.strip().str.capitalize() == 'Yes'
df_main = df_main.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

df_poi = safe_read(os.path.join(vip_folder, "POI.csv"))
df_poi.columns = df_poi.columns.str.strip()

# 2. CLUSTERING LOGIC
vital_pois = ["Bandar Udara Internasional Soekarno Hatta", "Semanggi", "KPPTI"]

def apply_vital_expansion(data, poi_data, region_name, hard_cap=200):
    data = data.copy()
    data['final_cluster_id'] = -1
    data['Covered_POI'] = "None"
    cluster_counter = 0
    if 'Region' in poi_data.columns:
        regional_pois = poi_data[poi_data['Region'].str.strip().str.upper() == region_name.strip().upper()].copy()
    else:
        reg_lat, reg_lon = data['Lat'].mean(), data['Lon'].mean()
        poi_data['temp_dist'] = np.sqrt((poi_data['Lat'] - reg_lat)**2 + (poi_data['Lon'] - reg_lon)**2)
        regional_pois = poi_data[poi_data['temp_dist'] < 0.5].copy()
    
    regional_pois['is_vital'] = regional_pois['POI'].isin(vital_pois)
    regional_pois = regional_pois.sort_values('is_vital', ascending=False)
    
    for _, poi in regional_pois.iterrows():
        lat_p, lon_p = poi['Lat'], poi['Lon']
        unassigned = data['final_cluster_id'] == -1
        if not unassigned.any(): break
        temp = data[unassigned].copy()
        temp['dist'] = np.sqrt((temp['Lat'] - lat_p)**2 + (temp['Lon'] - lon_p)**2)
        temp = temp.sort_values('dist')
        radius_limit = 999 if poi['is_vital'] else 0.018
        valid = temp[temp['dist'] <= radius_limit]
        if not valid.empty:
            indices = valid.head(hard_cap).index
            data.loc[indices, 'final_cluster_id'] = cluster_counter
            data.loc[indices, 'Covered_POI'] = poi['POI']
            cluster_counter += 1
    return data, cluster_counter

def process_residual(data, start_id, target_size=200):
    assigned = data[data['final_cluster_id'] != -1].copy()
    unassigned = data[data['final_cluster_id'] == -1].copy()
    if unassigned.empty: return assigned
    final_res, work_queue = [], [unassigned]
    curr_id = start_id
    while work_queue:
        batch = work_queue.pop(0)
        if len(batch) <= target_size:
            batch = batch.copy()
            batch['final_cluster_id'] = curr_id
            final_res.append(batch)
            curr_id += 1
        else:
            n_split = int(np.ceil(len(batch) / target_size))
            km = KMeans(n_clusters=n_split, n_init=10, random_state=42)
            labels = km.fit_predict(batch[['Lat', 'Lon']])
            for i in range(n_split):
                sub = batch[labels == i]
                if not sub.empty: work_queue.append(sub)
    return pd.concat([assigned] + final_res)

# 3. SNAKE (SERPENTINE) SEQUENCING
def calculate_activation_sequence(df_regional):
    centroids = df_regional.groupby('final_cluster_id').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
    num_strips = 6
    centroids['strip'] = pd.qcut(centroids['Lon'], num_strips, labels=False, duplicates='drop')
    
    serpentine_path = []
    for s_id in sorted(centroids['strip'].unique()):
        strip_data = centroids[centroids['strip'] == s_id].copy()
        ascending_toggle = (s_id % 2 == 0)
        strip_data = strip_data.sort_values(by='Lat', ascending=ascending_toggle)
        serpentine_path.append(strip_data)
        
    sequence_df = pd.concat(serpentine_path).reset_index(drop=True)
    sequence_df['Activation_Order'] = sequence_df.index + 1
    return sequence_df[['final_cluster_id', 'Activation_Order', 'Lat', 'Lon']]

# 4. EXECUTION
regional_results = []
path_data_list = []
for region, group in df_main.groupby('Region'):
    df_exp, n_id = apply_vital_expansion(group, df_poi, region)
    df_reg_final = process_residual(df_exp, n_id)
    df_reg_final['final_cluster_id'] = df_reg_final['final_cluster_id'].apply(lambda x: f"{region}_{int(x+1):02d}")
    
    seq_map = calculate_activation_sequence(df_reg_final)
    path_data_list.append(seq_map)
    df_reg_final = df_reg_final.merge(seq_map[['final_cluster_id', 'Activation_Order']], on='final_cluster_id')
    regional_results.append(df_reg_final)

df_final = pd.concat(regional_results).reset_index(drop=True)

# 5. GEOSPATIAL EXPORTS (Shapefiles)
geometry = [Point(xy) for xy in zip(df_final['Lon'], df_final['Lat'])]
gdf_sites = gpd.GeoDataFrame(df_final, geometry=geometry, crs="EPSG:4326")
hulls = [gdf_sites[gdf_sites['final_cluster_id'] == cid].geometry.unary_union.convex_hull.buffer(0.007) for cid in gdf_sites['final_cluster_id'].unique()]
unified_mask = unary_union(hulls)
all_coords = gdf_sites.geometry.unary_union
vor_polys = voronoi_diagram(all_coords, envelope=all_coords.envelope.buffer(0.05))
vor_gdf = gpd.GeoDataFrame(geometry=list(vor_polys.geoms), crs="EPSG:4326")
joined = gpd.sjoin(vor_gdf, gdf_sites[['final_cluster_id', 'geometry']], how="inner", predicate="contains")
seamless_vor = joined.dissolve(by='final_cluster_id').reset_index()
seamless_vor['geometry'] = seamless_vor.geometry.intersection(unified_mask)

# 6. DATA EXPORTS (CSV & XLSX) - The ones you were looking for!
summary_table = df_final.groupby(['Region', 'final_cluster_id', 'Activation_Order']).agg(
    total_sites=('New Sites ID', 'count'), 
    hvc_count=('is_hvc', 'sum')
).reset_index().sort_values(['Region', 'Activation_Order'])

# Save Shapefile
seamless_vor.merge(summary_table[['final_cluster_id', 'Activation_Order']], on='final_cluster_id').to_file(os.path.join(border_folder, "5G_2_6_Border.shp"))

# Save Master CSV
df_final.to_csv(os.path.join(result_folder, "Clustered_Site_Details.csv"), index=False)

# Save Multi-Sheet Excel
with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report.xlsx"), engine='xlsxwriter') as writer:
    df_final.sort_values(['Region', 'Activation_Order']).to_excel(writer, sheet_name='Site Details', index=False)
    summary_table.to_excel(writer, sheet_name='Cluster Summary', index=False)

# 7. MAP GENERATION (With Legend & Toggle)
legend_html = """
{% macro html(this, kwargs) %}
<div style="position: fixed; bottom: 50px; left: 50px; width: 180px; height: 110px; border:2px solid grey; z-index:9999; 
    font-size:14px; background-color: white; opacity: 0.85; padding: 10px; border-radius: 10px; font-family: 'Arial';">
    <b>Site Type Legend</b><br>
    <i style="color: black;">&#9650;</i> P1 / HVC Site<br>
    <i style="color: black;">&#9679;</i> Normal Site<br>
    <hr style="margin: 5px 0;">
    <small>Line: Serpentine Path</small>
</div>
{% endmacro %}
"""
legend = MacroElement()
legend._template = Template(legend_html)

colors = list(mcolors.TABLEAU_COLORS.values()) + list(mcolors.XKCD_COLORS.values())
color_map = {cid: colors[i % len(colors)] for i, cid in enumerate(df_final['final_cluster_id'].unique())}
map_center = [df_final['Lat'].mean(), df_final['Lon'].mean()]

def get_cluster_group():
    group = folium.FeatureGroup(name="Cluster Points (Shapes by HVC)")
    for _, row in df_final.iterrows():
        cid, color = row['final_cluster_id'], color_map[row['final_cluster_id']]
        if row['is_hvc']:
            RegularPolygonMarker(location=[row['Lat'], row['Lon']], number_of_sides=3, radius=7, color=color, fill=True, fill_opacity=0.6).add_to(group)
        else:
            folium.CircleMarker(location=[row['Lat'], row['Lon']], radius=4, color=color, fill=True, fill_opacity=0.6).add_to(group)
    return group

def get_mobility_group():
    group = folium.FeatureGroup(name="Serpentine Mobility Path")
    for reg_path in path_data_list:
        pts = reg_path[['Lat', 'Lon']].values.tolist()
        folium.PolyLine(pts, color="#2c3e50", weight=5, opacity=0.85).add_to(group)
        for _, p in reg_path.iterrows():
            folium.Marker(location=[p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'''
                <div style="font-size: 8pt; color: white; background: #2c3e50; border-radius: 4px; width: 22px; height: 16px; 
                text-align: center; line-height: 16px; border: 1px solid white; font-weight: bold;">{int(p["Activation_Order"])}</div>''')).add_to(group)
    return group

# Save the 3 HTML Maps
for name, func_list in {
    "5G_Only_Cluster_Points.html": [get_cluster_group],
    "5G_Only_Mobility_Path.html": [get_mobility_group],
    "5G_Combined_Clustering_Map.html": [get_cluster_group, get_mobility_group]
}.items():
    m = folium.Map(location=map_center, zoom_start=11, tiles='CartoDB Positron')
    for func in func_list:
        func().add_to(m)
    if len(func_list) > 1:
        folium.LayerControl(collapsed=False).add_to(m)
    m.add_child(legend)
    m.save(os.path.join(result_folder, name))

# 8. FINAL STATS CONSOLE SUMMARY
print("\n" + "="*50)
print("      NR ASSESSMENT: 5G CLUSTERING ENGINE v3.3")
print("="*50)
print(f"Total Clusters Generated      : {len(summary_table)}")
print(f"Average Sites / Cluster       : {summary_table['total_sites'].mean():.1f}")
print(f"Minimum / Maximum Cluster Size: {summary_table['total_sites'].min()} / {summary_table['total_sites'].max()}")
print(f"Total HVC Sites (Triangles)   : {df_final['is_hvc'].sum()}")
print("-" * 50)
print("ALL EXPORTED FILES READY:")
print(f"1. Excel Report (.xlsx) -> {os.path.join(result_folder, '5G_Cluster_Report.xlsx')}")
print(f"2. Master Site CSV (.csv) -> {os.path.join(result_folder, 'Clustered_Site_Details.csv')}")
print(f"3. GIS Borders (.shp)    -> {border_folder}")
print(f"4. Map: Points Only      -> 5G_Only_Cluster_Points.html")
print(f"5. Map: Path Only        -> 5G_Only_Mobility_Path.html")
print(f"6. Map: Combined+Toggle  -> 5G_Combined_Clustering_Map.html")
print("="*50)

C:\Users\m00811323\AppData\Local\Temp\ipykernel_28044\253231858.py:133: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hulls = [gdf_sites[gdf_sites['final_cluster_id'] == cid].geometry.unary_union.convex_hull.buffer(0.007) for cid in gdf_sites['final_cluster_id'].unique()]
C:\Users\m00811323\AppData\Local\Temp\ipykernel_28044\253231858.py:133: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hulls = [gdf_sites[gdf_sites['final_cluster_id'] == cid].geometry.unary_union.convex_hull.buffer(0.007) for cid in gdf_sites['final_cluster_id'].unique()]
C:\Users\m00811323\AppData\Local\Temp\ipykernel_28044\253231858.py:133: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hulls = [gdf_sites[gdf_sites['final_cluster_id'] == cid].geometry.unary_union.convex_hull.buffer(0.007) for cid in gdf_sites['final_cluster_id'].unique()]
C:\Users\m

C:\Users\m00811323\AppData\Local\Temp\ipykernel_28044\253231858.py:149: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  seamless_vor.merge(summary_table[['final_cluster_id', 'Activation_Order']], on='final_cluster_id').to_file(os.path.join(border_folder, "5G_2_6_Border.shp"))
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'final_cluster_id' to 'final_clus'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'index_right' to 'index_righ'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Activation_Order' to 'Activation'
  ogr_write(



      NR ASSESSMENT: 5G CLUSTERING ENGINE v3.3
Total Clusters Generated      : 83
Average Sites / Cluster       : 125.7
Minimum / Maximum Cluster Size: 8 / 200
Total HVC Sites (Triangles)   : 4960
--------------------------------------------------
ALL EXPORTED FILES READY:
1. Excel Report (.xlsx) -> 5G 2.6 site list\Result\5G_Cluster_Report.xlsx
2. Master Site CSV (.csv) -> 5G 2.6 site list\Result\Clustered_Site_Details.csv
3. GIS Borders (.shp)    -> 5G 2.6 site list\Result\Border
4. Map: Points Only      -> 5G_Only_Cluster_Points.html
5. Map: Path Only        -> 5G_Only_Mobility_Path.html
6. Map: Combined+Toggle  -> 5G_Combined_Clustering_Map.html


# Revise

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import folium
from folium.features import RegularPolygonMarker
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import random

# 1. PATH CONFIGURATION
revise_dir = os.path.join("5G 2.6 site list", "Result", "Revise")
revise_output_dir = os.path.join(revise_dir, "Result")

if not os.path.exists(revise_output_dir):
    os.makedirs(revise_output_dir)

csv_files = glob.glob(os.path.join(revise_dir, "*.csv"))

if not csv_files:
    print(f"Error: No CSV found in {revise_dir}")
else:
    file_path = csv_files[0]
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.strip()
    
    # 2. DATA PREP
    df = df.dropna(subset=['Lat', 'Lon', 'Region', 'Activation_Order'])
    df['is_hvc'] = df['is_hvc'].astype(str).str.upper().str.contains('TRUE')

    # 3. BEAUTIFIED SUMMARY CALCULATION
    summary_table = df.groupby(['Region', 'final_cluster_id', 'Activation_Order'], sort=False).agg(
        total_sites=('New Sites ID', 'count'), 
        hvc_count=('is_hvc', 'sum')
    ).reset_index()

    summary_table['HVC %'] = (summary_table['hvc_count'] / summary_table['total_sites'] * 100).round(1)
    
    def get_hvc_range(p):
        if p <= 30: return '0-30%'
        elif p <= 60: return '30-60%'
        else: return '60-100%'
    
    summary_table['HVC Density Range'] = summary_table['HVC %'].apply(get_hvc_range)
    summary_table['HVC %_Str'] = summary_table['HVC %'].astype(str) + "%"

    # 4. EXPORTS: CSV & BEAUTIFIED EXCEL
    df.to_csv(os.path.join(revise_output_dir, "Revised_Site_Details.csv"), index=False)
    
    xlsx_path = os.path.join(revise_output_dir, "Revised_Cluster_Report.xlsx")
    with pd.ExcelWriter(xlsx_path, engine='xlsxwriter') as writer:
        df.to_excel(writer, sheet_name='Site Details', index=False)
        summary_table.drop(columns=['HVC %']).to_excel(writer, sheet_name='Cluster Summary', index=False)
        
        workbook = writer.book
        summary_ws = writer.sheets['Cluster Summary']
        header_fmt = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'border': 1, 'fg_color': '#4F81BD', 'font_color': 'white'})
        body_fmt = workbook.add_format({'align': 'center', 'valign': 'vcenter', 'border': 1})
        
        for col_num, val in enumerate(summary_table.drop(columns=['HVC %']).columns):
            summary_ws.write(0, col_num, val, header_fmt)
        for row_num in range(len(summary_table)):
            for col_num in range(len(summary_table.drop(columns=['HVC %']).columns)):
                summary_ws.write(row_num + 1, col_num, summary_table.drop(columns=['HVC %']).iloc[row_num, col_num], body_fmt)
        summary_ws.set_column('A:H', 20)

    # 5. HIGH-CONTRAST GLOBAL COLOR ENGINE
    sw = df[['Lat', 'Lon']].min().values.tolist()
    ne = df[['Lat', 'Lon']].max().values.tolist()
    bounds = [sw, ne]

    unique_clusters = list(df['final_cluster_id'].unique())
    n_clusters = len(unique_clusters)
    
    # We combine multiple qualitative colormaps to get enough distinct colors for 35+ clusters
    qual_colors = []
    for cmap_name in ['tab20', 'tab20b', 'tab20c', 'Set1', 'Dark2', 'Paired']:
        cmap = plt.cm.get_cmap(cmap_name)
        qual_colors.extend([mcolors.to_hex(cmap(i)) for i in range(cmap.N)])
    
    # Ensure uniqueness and shuffle
    qual_colors = list(dict.fromkeys(qual_colors)) 
    random.shuffle(qual_colors)
    
    # If we need more than we have, we add Turbo at the end
    if n_clusters > len(qual_colors):
        extra_needed = n_clusters - len(qual_colors)
        extra_cmap = plt.cm.get_cmap('turbo', extra_needed)
        qual_colors.extend([mcolors.to_hex(extra_cmap(i)) for i in range(extra_needed)])

    color_map = {cid: qual_colors[i] for i, cid in enumerate(unique_clusters)}

    def add_legend(map_obj):
        legend_html = '''
        <div style="position: fixed; bottom: 50px; right: 50px; width: 180px; height: 110px; 
                    background-color: white; border:2px solid grey; z-index:9999; font-size:12px;
                    padding: 10px; border-radius: 5px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
            <b>Site Type Legend</b><br>
            <i class="fa fa-caret-up fa-2x" style="color:black; vertical-align: middle;"></i> 
            <span style="margin-left:8px;">HVC Site (Triangle)</span><br>
            <i class="fa fa-circle fa-1x" style="color:black; vertical-align: middle; margin-left:4px;"></i> 
            <span style="margin-left:11px;">Normal Site (Circle)</span><br>
            <hr style="margin: 5px 0;">
            <small>High-Contrast Global Palette</small>
        </div>
        '''
        map_obj.get_root().html.add_child(folium.Element(legend_html))

    def get_cluster_group():
        group = folium.FeatureGroup(name="Revised Points")
        for _, row in df.iterrows():
            cid, color = row['final_cluster_id'], color_map.get(row['final_cluster_id'], 'grey')
            popup = f"<b>ID:</b> {row['New Sites ID']}<br><b>Cluster:</b> {cid}"
            if row['is_hvc']:
                RegularPolygonMarker(location=[row['Lat'], row['Lon']], number_of_sides=3, radius=10, 
                                     color=color, fill=True, fill_color=color, fill_opacity=1.0, popup=popup).add_to(group)
            else:
                folium.CircleMarker(location=[row['Lat'], row['Lon']], radius=6, 
                                    color=color, fill=True, fill_color=color, fill_opacity=0.8, popup=popup).add_to(group)
        return group

    def get_mobility_group(m_obj):
        folium.map.CustomPane("path_pane", z_index=650).add_to(m_obj)
        group = folium.FeatureGroup(name="Mobility Path")
        path_df = df.groupby(['Region', 'final_cluster_id', 'Activation_Order'], sort=False).agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
        path_df['Order_Int'] = pd.to_numeric(path_df['Activation_Order'], errors='coerce')
        numeric_only = path_df.dropna(subset=['Order_Int']).copy()

        for region, reg_df in numeric_only.groupby('Region'):
            reg_df = reg_df.sort_values(by='Order_Int')
            pts = reg_df[['Lat', 'Lon']].values.tolist()
            if len(pts) > 1:
                folium.PolyLine(pts, color="#2c3e50", weight=5, opacity=0.8, pane="path_pane").add_to(group)
            
            for _, p in reg_df.iterrows():
                folium.Marker(location=[p['Lat'], p['Lon']], pane="path_pane", icon=folium.DivIcon(html=f'''
                    <div style="font-size: 8pt; color: white; background: #2c3e50; border-radius: 50%; width: 22px; 
                    height: 22px; line-height: 22px; text-align: center; border: 1px solid white; font-weight: bold;">
                    {int(p["Order_Int"])}</div>''')).add_to(group)
        return group

    # 6. SAVE ALL 4 MAPS
    map_configs = {
        "Revised_Points_Only.html": {"funcs": ["points"], "tiles": "CartoDB Positron", "attr": None},
        "Revised_Path_Only.html": {"funcs": ["paths"], "tiles": "CartoDB Positron", "attr": None},
        "Revised_Combined_Clustering.html": {"funcs": ["points", "paths"], "tiles": "CartoDB Positron", "attr": None},
        "Revised_Combined_Voyager_OSM.html": {
            "funcs": ["points", "paths"], 
            "tiles": "https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png",
            "attr": "CartoDB Voyager"
        }
    }
    
    for name, config in map_configs.items():
        m = folium.Map(tiles=config["tiles"], attr=config["attr"])
        if "points" in config["funcs"]: get_cluster_group().add_to(m)
        if "paths" in config["funcs"]: get_mobility_group(m).add_to(m)
        add_legend(m)
        if len(config["funcs"]) > 1: folium.LayerControl(collapsed=False).add_to(m)
        m.fit_bounds(bounds)
        m.save(os.path.join(revise_output_dir, name))

    print(f"Revision Complete. Used mixed-qualitative palette for maximum contrast across {n_clusters} clusters.")

C:\Users\m00811323\AppData\Local\Temp\ipykernel_27984\8312638.py:78: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap(cmap_name)


Revision Complete. Used mixed-qualitative palette for maximum contrast across 83 clusters.


# Nano Cluster

In [2]:
import pandas as pd
import numpy as np
import glob
import os
import folium
from folium.features import RegularPolygonMarker
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors

# 1. SETUP & DATA LOADING
base_folder = "5G 2.6 site list"
vip_folder = "VIP"
result_folder = os.path.join(base_folder, "Result")

if not os.path.exists(result_folder):
    os.makedirs(result_folder)

def safe_read(path):
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding='latin1')

csv_files = [f for f in glob.glob(os.path.join(base_folder, "*.csv")) if "Result" not in f and "VIP" not in f]
if not csv_files:
    raise FileNotFoundError("Could not find the site list CSV!")

df_main = safe_read(csv_files[0])
df_main.columns = df_main.columns.str.strip()
df_main['is_hvc'] = df_main['HVC'].astype(str).str.strip().str.capitalize() == 'Yes'
df_main = df_main.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

df_poi = safe_read(os.path.join(vip_folder, "POI.csv"))
df_poi.columns = df_poi.columns.str.strip()

# 2. HIERARCHICAL CLUSTERING LOGIC
def process_hierarchical_region(df_region, df_poi, region_name):
    data = df_region.copy()
    data['nano_temp_id'] = -1
    nano_counter = 1
    
    # --- A. POI NANO SEEDING ---
    if 'Region' in df_poi.columns:
        regional_pois = df_poi[df_poi['Region'].str.strip().str.upper() == region_name.strip().upper()].copy()
    else:
        reg_lat, reg_lon = data['Lat'].mean(), data['Lon'].mean()
        df_poi['temp_dist'] = np.sqrt((df_poi['Lat'] - reg_lat)**2 + (df_poi['Lon'] - reg_lon)**2)
        regional_pois = df_poi[df_poi['temp_dist'] < 0.3].copy()
    
    for _, poi in regional_pois.iterrows():
        unassigned = data['nano_temp_id'] == -1
        if not unassigned.any(): break
        temp = data[unassigned].copy()
        temp['dist'] = np.sqrt((temp['Lat'] - poi['Lat'])**2 + (temp['Lon'] - poi['Lon'])**2)
        indices = temp.sort_values('dist').head(25).index
        data.loc[indices, 'nano_temp_id'] = nano_counter
        nano_counter += 1

    # --- B. RESIDUAL NANO CLUSTERING ---
    unassigned_mask = data['nano_temp_id'] == -1
    if unassigned_mask.any():
        residual_data = data[unassigned_mask].copy()
        n_nanos_needed = int(np.ceil(len(residual_data) / 25))
        km_nano = KMeans(n_clusters=n_nanos_needed, n_init=10, random_state=42)
        data.loc[unassigned_mask, 'nano_temp_id'] = km_nano.fit_predict(residual_data[['Lat', 'Lon']]) + nano_counter

    # --- C. BIG CLUSTER AGGREGATION (Cap 200) ---
    n_big_clusters = int(np.ceil(len(data) / 200))
    nano_centroids = data.groupby('nano_temp_id').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
    km_big = KMeans(n_clusters=n_big_clusters, n_init=10, random_state=42)
    nano_centroids['big_cluster_num'] = km_big.fit_predict(nano_centroids[['Lat', 'Lon']]) + 1
    
    data = data.merge(nano_centroids[['nano_temp_id', 'big_cluster_num']], on='nano_temp_id')
    data['final_cluster_id'] = data['big_cluster_num'].apply(lambda x: f"{region_name}_{x:02d}")
    
    # --- D. LOCAL SERPENTINE MOBILITY ---
    final_data_list = []
    for big_id, group in data.groupby('final_cluster_id'):
        sub_nanos = group.groupby('nano_temp_id').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
        num_strips = 3 if len(sub_nanos) > 3 else 1
        sub_nanos['strip'] = pd.qcut(sub_nanos['Lon'], num_strips, labels=False, duplicates='drop')
        
        snake_ordered = []
        for s_id in sorted(sub_nanos['strip'].unique()):
            strip_data = sub_nanos[sub_nanos['strip'] == s_id].copy()
            snake_ordered.append(strip_data.sort_values('Lat', ascending=(s_id % 2 == 0)))
        
        ordered_sub = pd.concat(snake_ordered).reset_index(drop=True)
        ordered_sub['nano_idx'] = ordered_sub.index + 1
        
        group = group.merge(ordered_sub[['nano_temp_id', 'nano_idx']], on='nano_temp_id')
        group['nano_cluster_id'] = big_id + "_N" + group['nano_idx'].astype(str).str.zfill(2)
        group['Activation_Order'] = group['nano_idx']
        final_data_list.append(group)

    return pd.concat(final_data_list).drop(columns=['nano_temp_id', 'big_cluster_num', 'nano_idx'])

# 3. EXECUTION
all_results = []
for region, group in df_main.groupby('Region'):
    print(f"Processing: {region}...")
    all_results.append(process_hierarchical_region(group, df_poi, region))

df_final = pd.concat(all_results).reset_index(drop=True)

# 4. EXPORTS (EXCEL & CSV)
summary_table = df_final.groupby(['Region', 'final_cluster_id', 'nano_cluster_id', 'Activation_Order']).agg(
    total_sites=('New Sites ID', 'count'),
    hvc_count=('is_hvc', 'sum')
).reset_index().sort_values(['Region', 'final_cluster_id', 'Activation_Order'])

df_final.to_csv(os.path.join(result_folder, "Clustered_Site_Details.csv"), index=False)

with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report.xlsx"), engine='xlsxwriter') as writer:
    df_final.to_excel(writer, sheet_name='Site Details', index=False)
    summary_table.to_excel(writer, sheet_name='Cluster Summary', index=False)

# 5. MAP GENERATION (NAV STYLE)
nav_grey = '#2C3E50'
colors = list(mcolors.TABLEAU_COLORS.values()) + list(mcolors.XKCD_COLORS.values())
color_map = {cid: colors[i % len(colors)] for i, cid in enumerate(df_final['final_cluster_id'].unique())}

m = folium.Map(tiles='CartoDB Positron')
sw = df_final[['Lat', 'Lon']].min().values.tolist()
ne = df_final[['Lat', 'Lon']].max().values.tolist()
m.fit_bounds([sw, ne])

# --- LAYER 1: SITES (With Popups) ---
site_layer = folium.FeatureGroup(name="Sites", show=True)
for _, row in df_final.iterrows():
    color = color_map[row['final_cluster_id']]
    popup_html = f"""
    <div style="font-family: sans-serif; font-size: 10pt; width: 200px;">
        <b>Site ID:</b> {row['New Sites ID']}<br>
        <b>Name:</b> {row['New Site Name']}<br>
        <hr style="margin: 5px 0;">
        <b>Nano Cluster:</b> {row['nano_cluster_id']}<br>
        <b>Cluster:</b> {row['final_cluster_id']}<br>
        <b>Sequence:</b> {int(row['Activation_Order'])}
    </div>"""
    
    if str(row['Tag']).strip().capitalize() == 'Coverage':
        RegularPolygonMarker(location=[row['Lat'], row['Lon']], number_of_sides=3, radius=6, 
                             color=color, weight=0.5, fill=True, fill_opacity=0.3, popup=popup_html).add_to(site_layer)
    else:
        folium.CircleMarker(location=[row['Lat'], row['Lon']], radius=3.5, 
                            color=color, weight=0.5, fill=True, fill_opacity=0.3, popup=popup_html).add_to(site_layer)

# --- LAYER 2: NANO PATHS (Perfect Center with Flexbox) ---
nano_path_layer = folium.FeatureGroup(name="Nano Mobility Path", show=True)
for cid, group in df_final.groupby('final_cluster_id'):
    path_pts = group.groupby('nano_cluster_id').agg({'Lat':'mean', 'Lon':'mean', 'Activation_Order':'first'}).sort_values('Activation_Order')
    pts = path_pts[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color=nav_grey, weight=3, opacity=0.8).add_to(nano_path_layer)
    for _, p in path_pts.iterrows():
        # Flexbox Technique to force text to dead center
        folium.Marker(
            location=[p['Lat'], p['Lon']],
            icon=folium.DivIcon(
                icon_size=(16,16),
                icon_anchor=(8,8),
                html=f'''<div style="font-family: Arial; font-size: 8pt; font-weight: bold; color: white; background-color: {nav_grey}; 
                             width: 16px; height: 16px; border-radius: 50%; border: 1px solid white; 
                             display: flex; align-items: center; justify-content: center; box-sizing: border-box; overflow: hidden;">
                             {int(p["Activation_Order"])}</div>''')
        ).add_to(nano_path_layer)

# --- LAYER 3: BIG PATHS (Numbers 1, 2, 3 only) ---
big_path_layer = folium.FeatureGroup(name="Big Cluster Path", show=False)
# Sequence big clusters by their longitude (West to East) across the region
big_centers = df_final.groupby(['Region', 'final_cluster_id']).agg({'Lat':'mean', 'Lon':'mean'}).reset_index().sort_values(['Region', 'Lon'])
for region, r_group in big_centers.groupby('Region'):
    r_pts = r_group[['Lat', 'Lon']].values.tolist()
    if len(r_pts) > 1:
        folium.PolyLine(r_pts, color=nav_grey, weight=4, opacity=0.9).add_to(big_path_layer)
    
    # Use index + 1 for simple 1, 2, 3 numbering
    for i, (_, p) in enumerate(r_group.iterrows()):
        folium.Marker(
            location=[p['Lat'], p['Lon']],
            icon=folium.DivIcon(
                icon_size=(18,18),
                icon_anchor=(9,9),
                html=f'''<div style="font-family: Arial; font-size: 8pt; font-weight: bold; color: white; background-color: {nav_grey}; 
                             width: 18px; height: 18px; border-radius: 50%; border: 1px solid white; 
                             display: flex; align-items: center; justify-content: center; box-sizing: border-box; 
                             box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
                             {i+1}</div>''') # Use i+1 to number them sequentially
        ).add_to(big_path_layer)

site_layer.add_to(m); nano_path_layer.add_to(m); big_path_layer.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Combined_Clustering_Map.html"))

# 6. PROFESSIONAL SUMMARY
reg_summary = df_final.groupby('Region').agg(Big=('final_cluster_id', 'nunique'), Nano=('nano_cluster_id', 'nunique'), Sites=('New Sites ID', 'count')).reset_index()
print("\n" + "="*75)
print(f"{'REGION':<20} | {'BIG CLUSTERS':<15} | {'NANO CLUSTERS':<15} | {'SITES':<10}")
print("-" * 75)
for _, row in reg_summary.iterrows():
    print(f"{row['Region']:<20} | {row['Big']:<15} | {row['Nano']:<15} | {row['Sites']:<10}")
print("-" * 75)
print(f"Global Averages: {df_final.groupby(['Region', 'final_cluster_id']).size().mean():.1f} sites/Big | {df_final.groupby(['Region', 'final_cluster_id', 'nano_cluster_id']).size().mean():.1f} sites/Nano")
print(f"Reports saved to: {result_folder}")
print("="*75)

Processing: BALI...
Processing: CENTRAL JAVA...
Processing: EAST JAVA...
Processing: JABO...
Processing: SULAWESI...
Processing: WEST JAVA...

REGION               | BIG CLUSTERS    | NANO CLUSTERS   | SITES     
---------------------------------------------------------------------------
BALI                 | 3               | 20              | 478       
CENTRAL JAVA         | 5               | 33              | 807       
EAST JAVA            | 10              | 79              | 1959      
JABO                 | 24              | 189             | 4705      
SULAWESI             | 5               | 37              | 909       
WEST JAVA            | 7               | 50              | 1245      
---------------------------------------------------------------------------
Global Averages: 187.1 sites/Big | 24.8 sites/Nano
Reports saved to: 5G 2.6 site list\Result


# Final

In [26]:
import pandas as pd
import numpy as np
import glob
import os
import folium
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import voronoi_diagram
from folium.features import RegularPolygonMarker
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors

# 1. SETUP
base_folder = "5G 2.6 site list"
result_folder = os.path.join(base_folder, "Result")
os.makedirs(result_folder, exist_ok=True)

def safe_read(path):
    try: return pd.read_csv(path)
    except: return pd.read_csv(path, encoding='latin1')

csv_files = [f for f in glob.glob(os.path.join(base_folder, "*.csv")) if "Result" not in f]
df_main = safe_read(csv_files[0])
df_main.columns = df_main.columns.str.strip()
df_main['is_hvc'] = df_main['HVC'].astype(str).str.strip().str.capitalize() == 'Yes'
df_main = df_main.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 2. ADAPTIVE SEQUENCING
def apply_adaptive_sequence(cents, seq_col, force_urban=False):
    lon_range = cents['Lon'].max() - cents['Lon'].min()
    lat_range = cents['Lat'].max() - cents['Lat'].min()
    is_urban = force_urban or (len(cents) >= 10 and lon_range < 1.0)
    
    if is_urban:
        cents['col'] = pd.qcut(cents['Lon'], q=min(8, len(cents)), labels=False, duplicates='drop')
        zigzag = []
        for i, col_idx in enumerate(sorted(cents['col'].unique())):
            strip = cents[cents['col'] == col_idx].copy()
            strip = strip.sort_values('Lat', ascending=(i % 2 != 0))
            zigzag.append(strip)
        res = pd.concat(zigzag).reset_index(drop=True)
    else:
        if lon_range > lat_range:
            res = cents.sort_values(['Lon', 'Lat']).reset_index(drop=True)
        else:
            res = cents.sort_values(['Lat', 'Lon'], ascending=[False, True]).reset_index(drop=True)
            
    res[seq_col] = range(1, len(res) + 1)
    return res

# 3. HIERARCHICAL PROCESSING
def process_hierarchy(df_reg, region_name):
    is_jabo_reg = region_name.strip().upper() in ["JABO", "JAKARTA", "JABODETABEK"]
    
    # Big Clusters (Target 200)
    n_big = max(1, int(np.ceil(len(df_reg) / 195))) # Slightly lower denominator to force smaller clusters
    km_big = KMeans(n_clusters=n_big, n_init=15, random_state=42)
    df_reg['big_label'] = km_big.fit_predict(df_reg[['Lat', 'Lon']])
    bcents = df_reg.groupby('big_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
    bcents = apply_adaptive_sequence(bcents, 'big_seq', force_urban=is_jabo_reg)
    df_reg = df_reg.merge(bcents[['big_label', 'big_seq']], on='big_label')
    df_reg['big_cluster_id'] = df_reg['big_seq'].apply(lambda x: f"{region_name}_{x:02d}")
    
    final_dfs, nano_paths = [], []
    for bid, group in df_reg.groupby('big_cluster_id'):
        # Nano Clusters (Target 30)
        n_nano = max(1, int(np.ceil(len(group) / 28)))
        km_nano = KMeans(n_clusters=n_nano, n_init=15, random_state=42)
        group['nano_label'] = km_nano.fit_predict(group[['Lat', 'Lon']])
        ncents = group.groupby('nano_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
        ncents = apply_adaptive_sequence(ncents, 'nano_seq', force_urban=True)
        group = group.merge(ncents[['nano_label', 'nano_seq']], on='nano_label')
        group['nano_cluster_id'] = group.apply(lambda x: f"{bid}_N{int(x['nano_seq']):02d}", axis=1)
        final_dfs.append(group.drop(columns=['big_label', 'nano_label']))
        nano_paths.append(ncents.sort_values('nano_seq'))
        
    return pd.concat(final_dfs), bcents.sort_values('big_seq'), nano_paths

# EXECUTION
all_data, b_paths_all, n_paths_all = [], [], []
for region, group in df_main.groupby('Region'):
    df_res, b_p, n_ps = process_hierarchy(group, region)
    all_data.append(df_res)
    b_paths_all.append(b_p)
    n_paths_all.extend(n_ps)
df_final = pd.concat(all_data).reset_index(drop=True)

# 4. EXPORTS (Master CSV & Nano-Level XLSX Summary)
df_final.to_csv(os.path.join(result_folder, "5G_Master_Data.csv"), index=False)

# Creating Nano-Level Summary
summary_nano = df_final.groupby(['Region', 'big_cluster_id', 'nano_cluster_id']).agg(
    Total_Sites=('New Sites ID', 'count'),
    HVC_Sites=('is_hvc', 'sum'),
    Avg_Lat=('Lat', 'mean'),
    Avg_Lon=('Lon', 'mean')
).reset_index()

with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report.xlsx")) as writer:
    df_final.to_excel(writer, sheet_name='Site Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano Cluster Summary', index=False)

# 5. MAPPING (Interlocking Borders & Unified Paths)
geometry = [Point(xy) for xy in zip(df_final['Lon'], df_final['Lat'])]
gdf_sites = gpd.GeoDataFrame(df_final, geometry=geometry, crs="EPSG:4326")

def get_interlocking_gdf(gdf):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    joined = gpd.sjoin(vor_gdf, gdf[['big_cluster_id', 'geometry']], how="inner", predicate="contains")
    dissolved = joined.dissolve(by='big_cluster_id').reset_index()
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    return dissolved

interlocking_borders = get_interlocking_gdf(gdf_sites)

m = folium.Map(location=[df_main['Lat'].mean(), df_main['Lon'].mean()], zoom_start=10, tiles='CartoDB Positron')
lyr_border = folium.FeatureGroup(name="Interlocking Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Sites", show=True)
lyr_nano = folium.FeatureGroup(name="Nano Mobility", show=True)
lyr_big = folium.FeatureGroup(name="Big Cluster Mobility", show=False)

colors = list(mcolors.TABLEAU_COLORS.values()) * 25
color_map = {cid: colors[i % len(colors)] for i, cid in enumerate(df_final['big_cluster_id'].unique())}

for _, row in interlocking_borders.iterrows():
    c = color_map.get(row['big_cluster_id'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 1.2, 'fillOpacity': 0.25}).add_to(lyr_border)

for _, r in df_final.iterrows():
    c = color_map[r['big_cluster_id']]
    if r['is_hvc']: RegularPolygonMarker([r['Lat'], r['Lon']], number_of_sides=3, radius=6, color=c, fill=True).add_to(lyr_sites)
    else: folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True).add_to(lyr_sites)

def draw_path(df, seq, layer, weight, color):
    pts = df[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1).add_to(layer)
    for _, p in df.iterrows():
        folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'''<div style="font-size: 7pt; color: white; background: {color}; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p[seq])}</div>''')).add_to(layer)

for bp in b_paths_all: draw_path(bp, 'big_seq', lyr_big, 4, "#1A1A1A")
for np in n_paths_all: draw_path(np, 'nano_seq', lyr_nano, 3, "#000000")

lyr_border.add_to(m); lyr_sites.add_to(m); lyr_nano.add_to(m); lyr_big.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Final_Analysis_Map.html"))
print("Done! Summary updated to Nano Cluster level.")

C:\Users\m00811323\AppData\Local\Temp\ipykernel_24840\2233884539.py:99: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report.xlsx")) as writer:
C:\Users\m00811323\AppData\Local\Temp\ipykernel_24840\2233884539.py:108: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union


Done! Summary updated to Nano Cluster level.


# Final 0.2 (almost but the split)

In [27]:
import pandas as pd
import numpy as np
import glob
import os
import folium
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import voronoi_diagram
from folium.features import RegularPolygonMarker
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors

# 1. SETUP
base_folder = "5G 2.6 site list"
result_folder = os.path.join(base_folder, "Result")
os.makedirs(result_folder, exist_ok=True)

def safe_read(path):
    try: return pd.read_csv(path)
    except: return pd.read_csv(path, encoding='latin1')

csv_files = [f for f in glob.glob(os.path.join(base_folder, "*.csv")) if "Result" not in f]
df_main = safe_read(csv_files[0])
df_main.columns = df_main.columns.str.strip()
df_main['is_hvc'] = df_main['HVC'].astype(str).str.strip().str.capitalize() == 'Yes'
df_main = df_main.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 2. ADAPTIVE SEQUENCING
def apply_adaptive_sequence(cents, seq_col, force_urban=False):
    if len(cents) <= 1:
        cents[seq_col] = 1
        return cents
    lon_range = cents['Lon'].max() - cents['Lon'].min()
    lat_range = cents['Lat'].max() - cents['Lat'].min()
    is_urban = force_urban or (len(cents) >= 10 and lon_range < 1.0)
    
    if is_urban:
        cents['col'] = pd.qcut(cents['Lon'], q=min(8, len(cents)), labels=False, duplicates='drop')
        zigzag = []
        for i, col_idx in enumerate(sorted(cents['col'].unique())):
            strip = cents[cents['col'] == col_idx].copy()
            strip = strip.sort_values('Lat', ascending=(i % 2 != 0))
            zigzag.append(strip)
        res = pd.concat(zigzag).reset_index(drop=True)
    else:
        if lon_range > lat_range:
            res = cents.sort_values(['Lon', 'Lat']).reset_index(drop=True)
        else:
            res = cents.sort_values(['Lat', 'Lon'], ascending=[False, True]).reset_index(drop=True)
    res[seq_col] = range(1, len(res) + 1)
    return res

# 3. RECURSIVE K-MEANS (Ensures < 200 sites while keeping shapes clean)
def balanced_clustering(df, target_size, label_prefix):
    n_clusters = max(1, int(np.ceil(len(df) / target_size)))
    km = KMeans(n_clusters=n_clusters, n_init=15, random_state=42)
    df[label_prefix] = km.fit_predict(df[['Lat', 'Lon']])
    
    # Check for overfilled clusters and re-split them
    final_labels = []
    current_max_label = df[label_prefix].max()
    
    for lbl, group in df.groupby(label_prefix):
        if len(group) > 200:
            # Force split the overfilled cluster
            sub_n = int(np.ceil(len(group) / 190))
            sub_km = KMeans(n_clusters=sub_n, n_init=10, random_state=42)
            group[label_prefix] = sub_km.fit_predict(group[['Lat', 'Lon']]) + (current_max_label + 1)
            current_max_label = group[label_prefix].max()
        final_labels.append(group)
        
    return pd.concat(final_labels).reset_index(drop=True)

# 4. HIERARCHICAL PROCESSING
def process_hierarchy(df_reg, region_name):
    is_jabo_reg = region_name.strip().upper() in ["JABO", "JAKARTA", "JABODETABEK"]
    
    # Big Clusters with Recursive Split
    df_reg = balanced_clustering(df_reg, 195, 'big_label')
    
    bcents = df_reg.groupby('big_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
    bcents = apply_adaptive_sequence(bcents, 'big_seq', force_urban=is_jabo_reg)
    df_reg = df_reg.merge(bcents[['big_label', 'big_seq']], on='big_label')
    df_reg['big_cluster_id'] = df_reg['big_seq'].apply(lambda x: f"{region_name}_{x:02d}")
    
    final_dfs, nano_paths = [], []
    for bid, group in df_reg.groupby('big_cluster_id'):
        # Nano Clusters with Recursive Split
        group = balanced_clustering(group, 28, 'nano_label')
        ncents = group.groupby('nano_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
        ncents = apply_adaptive_sequence(ncents, 'nano_seq', force_urban=True)
        group = group.merge(ncents[['nano_label', 'nano_seq']], on='nano_label')
        group['nano_cluster_id'] = group.apply(lambda x: f"{bid}_N{int(x['nano_seq']):02d}", axis=1)
        final_dfs.append(group.drop(columns=['big_label', 'nano_label', 'big_seq']))
        nano_paths.append(ncents.sort_values('nano_seq'))
        
    return pd.concat(final_dfs), bcents.sort_values('big_seq'), nano_paths

# EXECUTION
all_data, b_paths_all, n_paths_all = [], [], []
for region, group in df_main.groupby('Region'):
    df_res, b_p, n_ps = process_hierarchy(group, region)
    all_data.append(df_res)
    b_paths_all.append(b_p)
    n_paths_all.extend(n_ps)
df_final = pd.concat(all_data).reset_index(drop=True)

# 5. EXPORTS
df_final.to_csv(os.path.join(result_folder, "5G_Master_Data.csv"), index=False)
summary_nano = df_final.groupby(['Region', 'big_cluster_id', 'nano_cluster_id']).size().reset_index(name='Total_Sites')

with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report.xlsx")) as writer:
    df_final.to_excel(writer, sheet_name='Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano Summary', index=False)

# 6. MAPPING (Voronoi Interlocking)
geometry = [Point(xy) for xy in zip(df_final['Lon'], df_final['Lat'])]
gdf_sites = gpd.GeoDataFrame(df_final, geometry=geometry, crs="EPSG:4326")

def get_interlocking_gdf(gdf):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    joined = gpd.sjoin(vor_gdf, gdf[['big_cluster_id', 'geometry']], how="inner", predicate="contains")
    dissolved = joined.dissolve(by='big_cluster_id').reset_index()
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    return dissolved

interlocking_borders = get_interlocking_gdf(gdf_sites)

m = folium.Map(location=[df_main['Lat'].mean(), df_main['Lon'].mean()], zoom_start=10, tiles='CartoDB Positron')
lyr_border = folium.FeatureGroup(name="Interlocking Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Sites", show=True)
lyr_nano = folium.FeatureGroup(name="Nano Mobility", show=True)

colors = list(mcolors.TABLEAU_COLORS.values()) * 40
color_map = {cid: colors[i % len(colors)] for i, cid in enumerate(df_final['big_cluster_id'].unique())}

for _, row in interlocking_borders.iterrows():
    c = color_map.get(row['big_cluster_id'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 1, 'fillOpacity': 0.25}).add_to(lyr_border)

for _, r in df_final.iterrows():
    c = color_map[r['big_cluster_id']]
    folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True).add_to(lyr_sites)

def draw_path(df, seq, layer, weight, color):
    pts = df[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1).add_to(layer)
    for _, p in df.iterrows():
        folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'''<div style="font-size: 7pt; color: white; background: {color}; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p[seq])}</div>''')).add_to(layer)

for np in n_paths_all: draw_path(np, 'nano_seq', lyr_nano, 3, "#000000")

lyr_border.add_to(m); lyr_sites.add_to(m); lyr_nano.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Final_Analysis_Map.html"))

print("Done. Used Recursive K-Means for perfect shapes and strict < 200 site limits.")

C:\Users\m00811323\AppData\Local\Temp\ipykernel_24840\1222820024.py:112: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report.xlsx")) as writer:
C:\Users\m00811323\AppData\Local\Temp\ipykernel_24840\1222820024.py:121: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union


Done. Used Recursive K-Means for perfect shapes and strict < 200 site limits.


# Final 0.3 Test

In [36]:
import pandas as pd
import numpy as np
import glob
import os
import folium
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import voronoi_diagram
from sklearn.cluster import KMeans
import matplotlib.colors as mcolors

# 1. SETUP
base_folder = "5G 2.6 site list"
result_folder = os.path.join(base_folder, "Result")
os.makedirs(result_folder, exist_ok=True)

def safe_read(path):
    try: return pd.read_csv(path)
    except: return pd.read_csv(path, encoding='latin1')

csv_files = [f for f in glob.glob(os.path.join(base_folder, "*.csv")) if "Result" not in f]
df_main = safe_read(csv_files[0])
df_main.columns = df_main.columns.str.strip()
df_main['is_hvc'] = df_main['HVC'].astype(str).str.strip().str.capitalize() == 'Yes'
df_main = df_main.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 2. THE SNAKE (ZIGZAG) LOGIC - RESTORED & LOCKED
def apply_adaptive_sequence(cents, seq_col, force_snake=False):
    if len(cents) <= 1:
        cents[seq_col] = 1
        return cents
    
    if force_snake:
        # JABO Snake: Divide into columns and flip Lat sorting every other column
        # Ensure we start from the West (lowest Lon)
        cents = cents.sort_values('Lon').reset_index(drop=True)
        # Use qcut to create balanced columns (adjusting 'q' based on cluster density)
        n_cols = min(6, len(cents))
        cents['col_bin'] = pd.qcut(cents['Lon'], q=n_cols, labels=False, duplicates='drop')
        
        zigzag_list = []
        for i, col_idx in enumerate(sorted(cents['col_bin'].unique())):
            strip = cents[cents['col_bin'] == col_idx].copy()
            # Alternate sorting: Even columns Top->Bottom, Odd columns Bottom->Top
            ascending_lat = (i % 2 != 0)
            strip = strip.sort_values('Lat', ascending=ascending_lat)
            zigzag_list.append(strip)
            
        res = pd.concat(zigzag_list).reset_index(drop=True)
    else:
        # Standard Sweep for Non-JABO
        res = cents.sort_values(['Lon', 'Lat']).reset_index(drop=True)
            
    res[seq_col] = range(1, len(res) + 1)
    return res

# 3. BALANCED CLUSTERING (Semanggi-Safe & 200/30 Caps)
def strict_balanced_clustering(df, target_size, max_cap, label_prefix):
    n_clusters = max(1, int(np.ceil(len(df) / (target_size - 5))))
    km = KMeans(n_clusters=n_clusters, init='k-means++', n_init=25, random_state=42)
    df[label_prefix] = km.fit_predict(df[['Lat', 'Lon']])
    
    final_dfs = []
    current_max = df[label_prefix].max()
    for lbl, group in df.groupby(label_prefix):
        if len(group) > max_cap:
            sub_n = int(np.ceil(len(group) / (target_size - 5)))
            sub_km = KMeans(n_clusters=sub_n, init='k-means++', n_init=15, random_state=42)
            group[label_prefix] = sub_km.fit_predict(group[['Lat', 'Lon']]) + (current_max + 1)
            current_max = group[label_prefix].max()
        final_dfs.append(group)
    return pd.concat(final_dfs).reset_index(drop=True)

# 4. HIERARCHICAL PROCESSING
def process_hierarchy(df_reg, region_name):
    is_jabo_reg = region_name.strip().upper() in ["JABO", "JAKARTA", "JABODETABEK"]
    
    # Big Clusters (Strictly < 200)
    df_reg = strict_balanced_clustering(df_reg, 185, 200, 'big_label')
    bcents = df_reg.groupby('big_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
    
    # APPLY THE SNAKE HERE
    bcents = apply_adaptive_sequence(bcents, 'big_seq', force_snake=is_jabo_reg)
    
    df_reg = df_reg.merge(bcents[['big_label', 'big_seq']], on='big_label')
    df_reg['big_cluster_id'] = df_reg['big_seq'].apply(lambda x: f"{region_name}_{x:02d}")
    
    final_dfs, nano_paths = [], []
    for bid, group in df_reg.groupby('big_cluster_id'):
        # Nano Clusters (Strictly < 30)
        group = strict_balanced_clustering(group, 25, 30, 'nano_label')
        ncents = group.groupby('nano_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
        # Nano level uses standard sweep to avoid local confusion
        ncents = apply_adaptive_sequence(ncents, 'nano_seq', force_snake=False)
        
        group = group.merge(ncents[['nano_label', 'nano_seq']], on='nano_label')
        group['nano_cluster_id'] = group.apply(lambda x: f"{bid}_N{int(x['nano_seq']):02d}", axis=1)
        final_dfs.append(group.drop(columns=['big_label', 'nano_label']))
        nano_paths.append(ncents.sort_values('nano_seq'))
        
    return pd.concat(final_dfs), bcents.sort_values('big_seq'), nano_paths

# EXECUTION
all_data, b_paths_all, n_paths_all = [], [], []
for region, group in df_main.groupby('Region'):
    df_res, b_p, n_ps = process_hierarchy(group, region)
    all_data.append(df_res)
    b_paths_all.append(b_p)
    n_paths_all.extend(n_ps)
df_final = pd.concat(all_data).reset_index(drop=True)

# 5. EXPORTS
df_final.to_csv(os.path.join(result_folder, "5G_Master_Data_Snake_Locked.csv"), index=False)
summary_nano = df_final.groupby(['Region', 'big_cluster_id', 'nano_cluster_id']).size().reset_index(name='Total_Sites')

with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report_Snake.xlsx")) as writer:
    df_final.to_excel(writer, sheet_name='Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano Summary', index=False)

# 6. MAPPING
geometry = [Point(xy) for xy in zip(df_final['Lon'], df_final['Lat'])]
gdf_sites = gpd.GeoDataFrame(df_final, geometry=geometry, crs="EPSG:4326")

def get_interlocking_gdf(gdf):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    joined = gpd.sjoin(vor_gdf, gdf[['big_cluster_id', 'geometry']], how="inner", predicate="contains")
    dissolved = joined.dissolve(by='big_cluster_id').reset_index()
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    return dissolved

interlocking_borders = get_interlocking_gdf(gdf_sites)
m = folium.Map(location=[-6.2191, 106.8121], zoom_start=11, tiles='CartoDB Positron')

lyr_border = folium.FeatureGroup(name="Interlocking Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Sites", show=True)
lyr_nano = folium.FeatureGroup(name="Nano Mobility", show=True)
lyr_big = folium.FeatureGroup(name="Big Cluster Snake Path", show=True)

colors = list(mcolors.TABLEAU_COLORS.values()) * 50
color_map = {cid: colors[i % len(colors)] for i, cid in enumerate(df_final['big_cluster_id'].unique())}

for _, row in interlocking_borders.iterrows():
    c = color_map.get(row['big_cluster_id'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 1, 'fillOpacity': 0.25}).add_to(lyr_border)

for _, r in df_final.iterrows():
    c = color_map[r['big_cluster_id']]
    folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True).add_to(lyr_sites)

def draw_path(df, seq, layer, weight, color):
    pts = df[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1).add_to(layer)
    for _, p in df.iterrows():
        folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'''<div style="font-size: 7pt; color: white; background: {color}; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p[seq])}</div>''')).add_to(layer)

for b_path in b_paths_all: draw_path(b_path, 'big_seq', lyr_big, 4.5, "#1C1C1C")
for n_path in n_paths_all: draw_path(n_path, 'nano_seq', lyr_nano, 3, "#000000")

lyr_border.add_to(m); lyr_sites.add_to(m); lyr_nano.add_to(m); lyr_big.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Final_Snake_Logic.html"))

print("Snake (Zigzag) sequence restored for JABO. Mapping verified.")

C:\Users\m00811323\AppData\Local\Temp\ipykernel_24840\3970509444.py:116: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Report_Snake.xlsx")) as writer:
C:\Users\m00811323\AppData\Local\Temp\ipykernel_24840\3970509444.py:125: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union


Snake (Zigzag) sequence restored for JABO. Mapping verified.


# Final Code

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import folium
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import voronoi_diagram
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import matplotlib.colors as mcolors

# 1. SETUP
base_folder = "5G 2.6 site list"
result_folder = os.path.join(base_folder, "Result")
os.makedirs(result_folder, exist_ok=True)

def safe_read(path):
    try: return pd.read_csv(path)
    except: return pd.read_csv(path, encoding='latin1')

csv_files = [f for f in glob.glob(os.path.join(base_folder, "*.csv")) if "Result" not in f]
if not csv_files:
    print("No CSV file found!"); exit()

df_main = safe_read(csv_files[0])
df_main.columns = df_main.columns.str.strip()
df_main = df_main.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 2. HYBRID SEQUENCING LOGIC (For Big Clusters)
def apply_hybrid_sequence(cents, seq_col, is_jabo=False):
    if len(cents) <= 1:
        cents[seq_col] = 1
        return cents
    if is_jabo:
        cents = cents.sort_values('Lon').reset_index(drop=True)
        n_cols = min(8, len(cents))
        cents['col_bin'] = pd.qcut(cents['Lon'], q=n_cols, labels=False, duplicates='drop')
        zigzag_list = []
        for i, col_idx in enumerate(sorted(cents['col_bin'].unique())):
            strip = cents[cents['col_bin'] == col_idx].copy()
            ascending_lat = (i % 2 != 0)
            strip = strip.sort_values('Lat', ascending=ascending_lat)
            zigzag_list.append(strip)
        res = pd.concat(zigzag_list).reset_index(drop=True)
    else:
        cents['start_score'] = cents['Lon'] - cents['Lat']
        start_idx = cents['start_score'].idxmin()
        unvisited = cents.index.tolist()
        current_idx = start_idx
        path = [current_idx]
        unvisited.remove(current_idx)
        coords = cents[['Lat', 'Lon']].values
        while unvisited:
            last_coord = coords[current_idx].reshape(1, -1)
            remaining_coords = coords[unvisited]
            distances = cdist(last_coord, remaining_coords)[0]
            nearest_in_unvisited = unvisited[np.argmin(distances)]
            path.append(nearest_in_unvisited)
            unvisited.remove(nearest_in_unvisited)
            current_idx = nearest_in_unvisited
        res = cents.loc[path].reset_index(drop=True)
    res[seq_col] = range(1, len(res) + 1)
    return res

# 2.1 NANO SEQUENCING (Snake Logic)
def apply_nano_snake_sequence(df_nodes, seq_col):
    if len(df_nodes) <= 1:
        df_nodes[seq_col] = 1
        return df_nodes
    df_nodes = df_nodes.sort_values('Lon').reset_index(drop=True)
    n_cols = max(3, min(6, len(df_nodes) // 4)) 
    df_nodes['col_bin'] = pd.qcut(df_nodes['Lon'], q=n_cols, labels=False, duplicates='drop')
    zigzag_list = []
    for i, col_idx in enumerate(sorted(df_nodes['col_bin'].unique())):
        strip = df_nodes[df_nodes['col_bin'] == col_idx].copy()
        ascending_lat = (i % 2 != 0)
        strip = strip.sort_values('Lat', ascending=ascending_lat)
        zigzag_list.append(strip)
    res = pd.concat(zigzag_list).reset_index(drop=True)
    res[seq_col] = range(1, len(res) + 1)
    return res.drop(columns=['col_bin'])

# 3. CLUSTERING
def strict_balanced_clustering(df, target_size, max_cap, label_prefix):
    n_clusters = max(1, int(np.ceil(len(df) / (target_size - 5))))
    km = KMeans(n_clusters=n_clusters, init='k-means++', n_init=25, random_state=42)
    df[label_prefix] = km.fit_predict(df[['Lat', 'Lon']])
    final_dfs = []
    current_max = df[label_prefix].max()
    for lbl, group in df.groupby(label_prefix):
        if len(group) > max_cap:
            sub_n = int(np.ceil(len(group) / (target_size - 5)))
            sub_km = KMeans(n_clusters=sub_n, init='k-means++', n_init=15, random_state=42)
            group[label_prefix] = sub_km.fit_predict(group[['Lat', 'Lon']]) + (current_max + 1)
            current_max = group[label_prefix].max()
        final_dfs.append(group)
    return pd.concat(final_dfs).reset_index(drop=True)

def process_hierarchy(df_reg, region_name):
    is_jabo_reg = region_name.strip().upper() in ["JABO", "JAKARTA", "JABODETABEK"]
    df_reg = strict_balanced_clustering(df_reg, 185, 200, 'big_label')
    bcents = df_reg.groupby('big_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
    bcents = apply_hybrid_sequence(bcents, 'big_seq', is_jabo=is_jabo_reg)
    
    df_reg = df_reg.merge(bcents[['big_label', 'big_seq']], on='big_label')
    df_reg['big_cluster_id'] = df_reg['big_seq'].apply(lambda x: f"{region_name}_{x:02d}")
    
    final_dfs, all_ncents = [], []
    for bid, group in df_reg.groupby('big_cluster_id'):
        group = strict_balanced_clustering(group, 25, 30, 'nano_label')
        ncents = group.groupby('nano_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
        ncents = apply_nano_snake_sequence(ncents, 'rel_nano_seq')
        
        group = group.merge(ncents[['nano_label', 'rel_nano_seq']], on='nano_label')
        group['nano_cluster_id'] = group.apply(lambda x: f"{bid}_N{int(x['rel_nano_seq']):02d}", axis=1)
        
        ncents_final = ncents.merge(group[['nano_label', 'nano_cluster_id']].drop_duplicates(), on='nano_label')
        all_ncents.append(ncents_final.sort_values('rel_nano_seq'))
        final_dfs.append(group.drop(columns=['big_label', 'nano_label']))
        
    return pd.concat(final_dfs), bcents.sort_values('big_seq'), all_ncents

# 4. EXECUTION
all_data, b_paths_all, n_paths_all = [], [], []
for region, group in df_main.groupby('Region'):
    df_res, b_p, n_ps = process_hierarchy(group, region)
    all_data.append(df_res)
    b_paths_all.append(b_p)
    n_paths_all.extend(n_ps)
df_final = pd.concat(all_data).reset_index(drop=True)

# 5. DATA EXPORTS (Report generation included here)
df_final.to_csv(os.path.join(result_folder, "5G_Master_Site_List.csv"), index=False)

summary_nano = df_final.groupby(['Region', 'big_cluster_id', 'nano_cluster_id']).size().reset_index(name='Total_Sites')
with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Analysis_Report.xlsx")) as writer:
    df_final.to_excel(writer, sheet_name='Site_Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano_Cluster_Summary', index=False)

# 6. GEOSPATIAL & VORONOI EXPORT
geometry = [Point(xy) for xy in zip(df_final['Lon'], df_final['Lat'])]
gdf_sites = gpd.GeoDataFrame(df_final, geometry=geometry, crs="EPSG:4326")

def generate_voronoi_shp(gdf, id_col, output_name):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    
    cols_to_keep = ['big_cluster_id', 'nano_cluster_id', 'big_seq', 'rel_nano_seq', 'geometry']
    joined = gpd.sjoin(vor_gdf, gdf[cols_to_keep], how="inner", predicate="contains")
    
    dissolved = joined.dissolve(by=id_col).reset_index()
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    dissolved.to_file(os.path.join(result_folder, output_name))
    return dissolved

print("Exporting Voronoi Shapefiles...")
big_borders = generate_voronoi_shp(gdf_sites, 'big_cluster_id', "Big_Cluster_Borders.shp")
nano_borders = generate_voronoi_shp(gdf_sites, 'nano_cluster_id', "Nano_Cluster_Borders.shp")

# 7. MAPPING
m = folium.Map(location=[df_final['Lat'].mean(), df_final['Lon'].mean()], zoom_start=11, tiles='CartoDB Positron')
lyr_big_vor = folium.FeatureGroup(name="Big Cluster Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Clusterized Sites", show=True)
lyr_big_path = folium.FeatureGroup(name="Big Cluster Sequence", show=True)
lyr_nano_path = folium.FeatureGroup(name="Nano Mobility Paths (Off)", show=False)

colors = list(mcolors.TABLEAU_COLORS.values()) * 50
big_color_map = {bid: colors[i % len(colors)] for i, bid in enumerate(df_final['big_cluster_id'].unique())}

for _, row in big_borders.iterrows():
    c = big_color_map.get(row['big_cluster_id'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 2, 'fillOpacity': 0.25}).add_to(lyr_big_vor)

for _, r in df_final.iterrows():
    c = big_color_map[r['big_cluster_id']]
    s_id = r.get('New Sites ID', r.get('Unique ID', 'N/A'))
    popup_html = f"<b>ID:</b> {s_id}<br><b>Big:</b> {r['big_cluster_id']}<br><b>Nano:</b> {r['nano_cluster_id']}"
    folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True, fill_opacity=0.8, weight=1, popup=folium.Popup(popup_html, max_width=200)).add_to(lyr_sites)

def draw_styled_path(df, seq, layer, weight, color, add_labels=False):
    pts = df[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1.0).add_to(layer)
    if add_labels:
        for _, p in df.iterrows():
            folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'<div style="font-size: 7pt; color: white; background: {color}; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p[seq])}</div>')).add_to(layer)

for b_p in b_paths_all: draw_styled_path(b_p, 'big_seq', lyr_big_path, 4.5, "#1C1C1C", add_labels=True)
for n_p in n_paths_all: draw_styled_path(n_p, 'rel_nano_seq', lyr_nano_path, 3, "#444444")

lyr_big_vor.add_to(m); lyr_sites.add_to(m); lyr_big_path.add_to(m); lyr_nano_path.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Cluster_Map_Styled.html"))

print("Excel Report, Shapefiles, and Map generated successfully.")

C:\Users\m00811323\AppData\Local\Temp\ipykernel_28772\1355638646.py:137: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Analysis_Report.xlsx")) as writer:


Exporting Voronoi Shapefiles...


C:\Users\m00811323\AppData\Local\Temp\ipykernel_28772\1355638646.py:146: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union
C:\Users\m00811323\AppData\Local\Temp\ipykernel_28772\1355638646.py:156: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  dissolved.to_file(os.path.join(result_folder, output_name))
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'big_cluster_id' to 'big_cluste'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'index_right' to 'index_righ'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'nano_cluster_id' to 'nano_clu

Excel Report, Shapefiles, and Map generated successfully.


# Manual Adjustment

In [1]:
import pandas as pd
import numpy as np
import os
import folium
import geopandas as gpd
import matplotlib.colors as mcolors
from shapely.geometry import Point
from shapely.ops import voronoi_diagram

# 1. SETUP PATHS
base_folder = "5G 2.6 site list"
revise_folder = os.path.join(base_folder, "Result", "Revise")
result_folder = os.path.join(revise_folder, "Result") 
os.makedirs(result_folder, exist_ok=True)

# 2. LOAD DATA
input_path = os.path.join(revise_folder, "5G_Master_Site_List.csv")
if not os.path.exists(input_path):
    print(f"Error: {input_path} not found."); exit()

df = pd.read_csv(input_path)
df = df.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 3. GEOSPATIAL PROCESSING & SPECIFIC POLYGON EXPORT
geometry = [Point(xy) for xy in zip(df['Lon'], df['Lat'])]
gdf_sites = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

def export_specific_border(gdf, output_name):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    
    # Calculate Total Sites per Nano Cluster
    site_counts = gdf.groupby('nano_cluster_id').size().reset_index(name='Total Sites')
    
    # Join site data to Voronoi polygons
    # Mapping your requested info:
    # ["Region", "Big Cluster ID", "Big Cluster Sequence", "Nano Cluster ID", "Nano Cluster Sequence", "Total Sites"]
    cols_map = {
        'Region': 'Region',
        'big_cluster_id': 'Big Cluster ID',
        'big_seq': 'Big Cluster Sequence',
        'nano_cluster_id': 'Nano Cluster ID',
        'rel_nano_seq': 'Nano Cluster Sequence'
    }
    
    # Select unique attributes per Nano Cluster
    attr_gdf = gdf[['Region', 'big_cluster_id', 'big_seq', 'nano_cluster_id', 'rel_nano_seq', 'geometry']].copy()
    attr_gdf = attr_gdf.rename(columns=cols_map)
    attr_gdf = attr_gdf.merge(site_counts, left_on='Nano Cluster ID', right_on='nano_cluster_id').drop(columns=['nano_cluster_id'])
    
    joined = gpd.sjoin(vor_gdf, attr_gdf, how="inner", predicate="contains")
    
    # Dissolve at Nano Cluster ID level
    dissolved = joined.dissolve(by='Nano Cluster ID').reset_index()
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    
    # Final column cleanup for export
    final_cols = ["Region", "Big Cluster ID", "Big Cluster Sequence", "Nano Cluster ID", "Nano Cluster Sequence", "Total Sites", "geometry"]
    dissolved = dissolved[final_cols]
    
    dissolved.to_file(os.path.join(result_folder, output_name))
    return dissolved

print("Exporting 5G 2.6 Border.shp...")
nano_borders = export_specific_border(gdf_sites, "5G 2.6 Border.shp")

# 4. EXPORT SUMMARY (XLSX)
summary_nano = df.groupby(['Region', 'big_cluster_id', 'nano_cluster_id']).size().reset_index(name='Total_Sites')
xlsx_path = os.path.join(result_folder, "5G_Final_Revised_Report.xlsx")
with pd.ExcelWriter(xlsx_path) as writer:
    df.to_excel(writer, sheet_name='Site_Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano_Cluster_Summary', index=False)

# 5. RE-RENDER PREMIUM MAP
m = folium.Map(location=[df['Lat'].mean(), df['Lon'].mean()], zoom_start=11, tiles='CartoDB Positron')

lyr_big_vor = folium.FeatureGroup(name="Big Cluster Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Clusterized Sites", show=True)
lyr_big_path = folium.FeatureGroup(name="Big Cluster Sequence", show=True)
lyr_nano_path = folium.FeatureGroup(name="Nano Mobility Paths (Off)", show=False)

colors = list(mcolors.TABLEAU_COLORS.values()) * 50
big_color_map = {bid: colors[i % len(colors)] for i, bid in enumerate(df['big_cluster_id'].unique())}

# Borders (Big Level for visualization)
big_borders = nano_borders.dissolve(by='Big Cluster ID').reset_index()
for _, row in big_borders.iterrows():
    c = big_color_map.get(row['Big Cluster ID'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 2, 'fillOpacity': 0.25}).add_to(lyr_big_vor)

# Sites + Popups
for _, r in df.iterrows():
    c = big_color_map[r['big_cluster_id']]
    s_id = r.get('New Sites ID', r.get('Unique ID', 'N/A'))
    popup_html = f"<b>ID:</b> {s_id}<br><b>Big:</b> {r['big_cluster_id']}<br><b>Nano:</b> {r['nano_cluster_id']}"
    folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True, fill_opacity=0.8, weight=1, popup=folium.Popup(popup_html, max_width=250)).add_to(lyr_sites)

# Path Styling
def draw_styled_path(df_sub, seq_col, layer, weight, color, add_labels=False):
    pts = df_sub.sort_values(seq_col)[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1.0).add_to(layer)
    if add_labels:
        for _, p in df_sub.iterrows():
            folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'<div style="font-size: 7pt; color: white; background: {color}; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p[seq_col])}</div>')).add_to(layer)

# Path Logic
big_cents = df.groupby('big_cluster_id').agg({'Lat':'mean', 'Lon':'mean', 'big_seq':'first'}).reset_index()
for region, reg_group in df.groupby('Region'):
    reg_cents = big_cents[big_cents['big_cluster_id'].str.startswith(region)]
    draw_styled_path(reg_cents, 'big_seq', lyr_big_path, 4.5, "#1C1C1C", add_labels=True)

nano_cents = df.groupby('nano_cluster_id').agg({'Lat':'mean', 'Lon':'mean', 'rel_nano_seq':'first', 'big_cluster_id':'first'}).reset_index()
for bid, bid_group in nano_cents.groupby('big_cluster_id'):
    draw_styled_path(bid_group, 'rel_nano_seq', lyr_nano_path, 3, "#444444")

lyr_big_vor.add_to(m); lyr_sites.add_to(m); lyr_big_path.add_to(m); lyr_nano_path.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Final_Revised_Map.html"))

print(f"Success! Polygon '5G 2.6 Border.shp' and map generated in: {result_folder}")

Exporting 5G 2.6 Border.shp...


C:\Users\m00811323\AppData\Local\Temp\ipykernel_40180\3284246810.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union
C:\Users\m00811323\AppData\Local\Temp\ipykernel_40180\3284246810.py:63: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  dissolved.to_file(os.path.join(result_folder, output_name))
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Big Cluster ID' to 'Big Cluste'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Big Cluster Sequence' to 'Big Clus_1'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Nano Cluster ID' to 'N

Success! Polygon '5G 2.6 Border.shp' and map generated in: 5G 2.6 site list\Result\Revise\Result


# Ngide

In [2]:
import pandas as pd
import numpy as np
import os
import folium
import geopandas as gpd
import matplotlib.colors as mcolors
from shapely.geometry import Point
from shapely.ops import voronoi_diagram

# 1. SETUP PATHS
base_folder = "5G 2.6 site list"
revise_folder = os.path.join(base_folder, "Result", "Revise")
result_folder = os.path.join(revise_folder, "Result") 
os.makedirs(result_folder, exist_ok=True)

# 2. LOAD DATA
input_path = os.path.join(revise_folder, "5G_Master_Site_List (Rev).csv")
if not os.path.exists(input_path):
    print(f"Error: {input_path} not found."); exit()

df = pd.read_csv(input_path)
df = df.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 3. GEOSPATIAL PROCESSING & SPECIFIC POLYGON EXPORT
geometry = [Point(xy) for xy in zip(df['Lon'], df['Lat'])]
gdf_sites = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

def export_specific_border(gdf, output_name):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    
    site_counts = gdf.groupby('nano_cluster_id').size().reset_index(name='Total Sites')
    
    cols_map = {
        'Region': 'Region',
        'big_cluster_id': 'Big Cluster ID',
        'big_seq': 'Big Cluster Sequence',
        'nano_cluster_id': 'Nano Cluster ID',
        'rel_nano_seq': 'Nano Cluster Sequence'
    }
    
    attr_gdf = gdf[['Region', 'big_cluster_id', 'big_seq', 'nano_cluster_id', 'rel_nano_seq', 'geometry']].copy()
    attr_gdf = attr_gdf.rename(columns=cols_map)
    attr_gdf = attr_gdf.merge(site_counts, left_on='Nano Cluster ID', right_on='nano_cluster_id').drop(columns=['nano_cluster_id'])
    
    joined = gpd.sjoin(vor_gdf, attr_gdf, how="inner", predicate="contains")
    dissolved = joined.dissolve(by='Nano Cluster ID').reset_index()
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    
    final_cols = ["Region", "Big Cluster ID", "Big Cluster Sequence", "Nano Cluster ID", "Nano Cluster Sequence", "Total Sites", "geometry"]
    dissolved = dissolved[final_cols]
    
    dissolved.to_file(os.path.join(result_folder, output_name))
    return dissolved

print("Exporting 5G 2.6 Border.shp...")
nano_borders = export_specific_border(gdf_sites, "5G 2.6 Border.shp")

# 4. EXPORT SUMMARY (XLSX)
summary_nano = df.groupby(['Region', 'big_cluster_id', 'nano_cluster_id']).size().reset_index(name='Total_Sites')
xlsx_path = os.path.join(result_folder, "5G_Final_Revised_Report.xlsx")
with pd.ExcelWriter(xlsx_path) as writer:
    df.to_excel(writer, sheet_name='Site_Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano_Cluster_Summary', index=False)

# 5. RE-RENDER PREMIUM MAP
m = folium.Map(location=[df['Lat'].mean(), df['Lon'].mean()], zoom_start=11, tiles='CartoDB Positron')

lyr_big_vor = folium.FeatureGroup(name="Big Cluster Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Clusterized Sites", show=True)
lyr_big_path = folium.FeatureGroup(name="Big Cluster Sequence", show=True)
lyr_nano_path = folium.FeatureGroup(name="Nano Mobility Paths (Off)", show=False)

colors = list(mcolors.TABLEAU_COLORS.values()) * 50
big_color_map = {bid: colors[i % len(colors)] for i, bid in enumerate(df['big_cluster_id'].unique())}

big_borders = nano_borders.dissolve(by='Big Cluster ID').reset_index()
for _, row in big_borders.iterrows():
    c = big_color_map.get(row['Big Cluster ID'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 2, 'fillOpacity': 0.25}).add_to(lyr_big_vor)

for _, r in df.iterrows():
    c = big_color_map[r['big_cluster_id']]
    s_id = r.get('New Sites ID', r.get('Unique ID', 'N/A'))
    popup_html = f"<b>ID:</b> {s_id}<br><b>Big:</b> {r['big_cluster_id']}<br><b>Nano:</b> {r['nano_cluster_id']}"
    folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True, fill_opacity=0.8, weight=1, popup=folium.Popup(popup_html, max_width=250)).add_to(lyr_sites)

# Standard path drawing helper
def draw_styled_path(pts, layer, weight, color):
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1.0).add_to(layer)

# Path Logic: Group by Region AND Area to keep it clean
big_cents = df.groupby(['Region', 'area', 'big_cluster_id']).agg({'Lat':'mean', 'Lon':'mean', 'big_seq':'first'}).reset_index()

for (region, area), group in big_cents.groupby(['Region', 'area']):
    pts = group.sort_values('big_seq')[['Lat', 'Lon']].values.tolist()
    draw_styled_path(pts, lyr_big_path, 4.5, "#1C1C1C")

# SPECIAL CASE: Manually connect all "17"s to "18" across areas in the same Region
for region, reg_group in big_cents.groupby('Region'):
    pts_17 = reg_group[reg_group['big_seq'] == 17][['Lat', 'Lon']].values.tolist()
    pts_18 = reg_group[reg_group['big_seq'] == 18][['Lat', 'Lon']].values.tolist()
    
    for p17 in pts_17:
        for p18 in pts_18:
            draw_styled_path([p17, p18], lyr_big_path, 4.5, "#1C1C1C")

# Add Labels
for _, p in big_cents.iterrows():
    folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'<div style="font-size: 7pt; color: white; background: #1C1C1C; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p["big_seq"])}</div>')).add_to(lyr_big_path)

# Nano Path Logic (Normal)
nano_cents = df.groupby('nano_cluster_id').agg({'Lat':'mean', 'Lon':'mean', 'rel_nano_seq':'first', 'big_cluster_id':'first'}).reset_index()
for bid, bid_group in nano_cents.groupby('big_cluster_id'):
    pts = bid_group.sort_values('rel_nano_seq')[['Lat', 'Lon']].values.tolist()
    draw_styled_path(pts, lyr_nano_path, 3, "#444444")

lyr_big_vor.add_to(m); lyr_sites.add_to(m); lyr_big_path.add_to(m); lyr_nano_path.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Final_Revised_Map.html"))

print(f"Success! Cleaned up map with special 17->18 connection generated in: {result_folder}")

Exporting 5G 2.6 Border.shp...


C:\Users\m00811323\AppData\Local\Temp\ipykernel_28772\1434754003.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union
C:\Users\m00811323\AppData\Local\Temp\ipykernel_28772\1434754003.py:55: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  dissolved.to_file(os.path.join(result_folder, output_name))
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Big Cluster ID' to 'Big Cluste'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Big Cluster Sequence' to 'Big Clus_1'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Nano Cluster ID' to 'N

KeyError: 'area'

In [1]:
import pandas as pd
import numpy as np
import os
import folium
import geopandas as gpd
import matplotlib.colors as mcolors
from shapely.geometry import Point
from shapely.ops import voronoi_diagram

# 1. SETUP PATHS
base_folder = "5G 2.6 site list"
revise_folder = os.path.join(base_folder, "Result", "Revise")
result_folder = os.path.join(revise_folder, "Result")
os.makedirs(result_folder, exist_ok=True)

# 2. LOAD DATA
input_path = os.path.join(revise_folder, "5G_Master_Site_List.csv")
if not os.path.exists(input_path):
    print(f"Error: {input_path} not found.")
    exit()

df = pd.read_csv(input_path)
df = df.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 3. GEOSPATIAL PROCESSING & SPECIFIC POLYGON EXPORT
geometry = [Point(xy) for xy in zip(df['Lon'], df['Lat'])]
gdf_sites = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

def export_specific_border(gdf, output_name):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    
    # Calculate Total Sites per Nano Cluster
    site_counts = gdf.groupby('nano_cluster_id').size().reset_index(name='Total Sites')
    
    # Mapping columns
    cols_map = {
        'Region': 'Region',
        'big_cluster_id': 'Big Cluster ID',
        'big_seq': 'Big Cluster Sequence',
        'nano_cluster_id': 'Nano Cluster ID',
        'rel_nano_seq': 'Nano Cluster Sequence'
    }
    
    attr_gdf = gdf[['Region', 'big_cluster_id', 'big_seq', 'nano_cluster_id', 'rel_nano_seq', 'geometry']].copy()
    attr_gdf = attr_gdf.rename(columns=cols_map)
    attr_gdf = attr_gdf.merge(site_counts, left_on='Nano Cluster ID', right_on='nano_cluster_id').drop(columns=['nano_cluster_id'])
    
    joined = gpd.sjoin(vor_gdf, attr_gdf, how="inner", predicate="contains")
    
    # Dissolve at Nano Cluster ID level
    dissolved = joined.dissolve(by='Nano Cluster ID').reset_index()
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    
    final_cols = ["Region", "Big Cluster ID", "Big Cluster Sequence", "Nano Cluster ID", "Nano Cluster Sequence", "Total Sites", "geometry"]
    dissolved = dissolved[final_cols]
    
    dissolved.to_file(os.path.join(result_folder, output_name))
    return dissolved

print("Exporting 5G 2.6 Border.shp...")
nano_borders = export_specific_border(gdf_sites, "5G 2.6 Border.shp")

# 4. EXPORT SUMMARY (XLSX)
summary_nano = df.groupby(['Region', 'big_cluster_id', 'nano_cluster_id']).size().reset_index(name='Total_Sites')
xlsx_path = os.path.join(result_folder, "5G_Final_Revised_Report.xlsx")
with pd.ExcelWriter(xlsx_path) as writer:
    df.to_excel(writer, sheet_name='Site_Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano_Cluster_Summary', index=False)

# 5. RE-RENDER PREMIUM MAP
m = folium.Map(location=[df['Lat'].mean(), df['Lon'].mean()], zoom_start=11, tiles='CartoDB Positron')

lyr_big_vor = folium.FeatureGroup(name="Big Cluster Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Clusterized Sites", show=True)
lyr_big_path = folium.FeatureGroup(name="Big Cluster Sequence", show=True)
lyr_nano_path = folium.FeatureGroup(name="Nano Mobility Paths (Off)", show=False)

# JABO Zone Color Mapping
zone_colors = {
    "WEST_JABO": "#1E90FF",   # Blue
    "SOUTH_JABO": "#FF4500",  # Red
    "EAST_JABO": "#2E8B57"    # Green
}

colors = list(mcolors.TABLEAU_COLORS.values()) * 50
big_color_map = {bid: colors[i % len(colors)] for i, bid in enumerate(df['big_cluster_id'].unique())}

# Borders (Big Level for visualization)
big_borders = nano_borders.dissolve(by='Big Cluster ID').reset_index()
for _, row in big_borders.iterrows():
    c = big_color_map.get(row['Big Cluster ID'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 2, 'fillOpacity': 0.25}).add_to(lyr_big_vor)

# Sites + Popups
for _, r in df.iterrows():
    c = big_color_map[r['big_cluster_id']]
    s_id = r.get('New Sites ID', r.get('Unique ID', 'N/A'))
    popup_html = f"<b>ID:</b> {s_id}<br><b>Big:</b> {r['big_cluster_id']}<br><b>Nano:</b> {r['nano_cluster_id']}"
    folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True, fill_opacity=0.8, weight=1, popup=folium.Popup(popup_html, max_width=250)).add_to(lyr_sites)

# Path Styling function
def draw_styled_path(df_sub, seq_col, layer, weight, color, add_labels=False):
    pts = df_sub.sort_values(seq_col)[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1.0).add_to(layer)
    if add_labels:
        for _, p in df_sub.iterrows():
            folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'<div style="font-size: 7pt; color: white; background: {color}; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p[seq_col])}</div>')).add_to(layer)

# Path Logic with Zone-Specific Colors
big_cents = df.groupby('big_cluster_id').agg({'Lat':'mean', 'Lon':'mean', 'big_seq':'first'}).reset_index()
for region, reg_group in df.groupby('Region'):
    # Select color based on JABO zone, default to dark grey
    path_color = zone_colors.get(region, "#1C1C1C")
    reg_cents = big_cents[big_cents['big_cluster_id'].str.startswith(region)]
    draw_styled_path(reg_cents, 'big_seq', lyr_big_path, 4.5, path_color, add_labels=True)

nano_cents = df.groupby('nano_cluster_id').agg({'Lat':'mean', 'Lon':'mean', 'rel_nano_seq':'first', 'big_cluster_id':'first'}).reset_index()
for bid, bid_group in nano_cents.groupby('big_cluster_id'):
    # Find the region for this specific big_cluster to keep colors consistent
    region_of_bid = df[df['big_cluster_id'] == bid]['Region'].iloc[0]
    nano_path_color = zone_colors.get(region_of_bid, "#444444")
    draw_styled_path(bid_group, 'rel_nano_seq', lyr_nano_path, 3, nano_path_color)

lyr_big_vor.add_to(m); lyr_sites.add_to(m); lyr_big_path.add_to(m); lyr_nano_path.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Final_Revised_Map.html"))

print(f"Success! Polygon '5G 2.6 Border.shp' and map generated in: {result_folder}")

Exporting 5G 2.6 Border.shp...


C:\Users\m00811323\AppData\Local\Temp\ipykernel_38460\4026176126.py:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union
C:\Users\m00811323\AppData\Local\Temp\ipykernel_38460\4026176126.py:60: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  dissolved.to_file(os.path.join(result_folder, output_name))
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Big Cluster ID' to 'Big Cluste'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Big Cluster Sequence' to 'Big Clus_1'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Nano Cluster ID' to 'N

Success! Polygon '5G 2.6 Border.shp' and map generated in: 5G 2.6 site list\Result\Revise\Result


# City Based

In [3]:
import pandas as pd
import numpy as np
import glob
import os
import re
import folium
import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import voronoi_diagram
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
import matplotlib.colors as mcolors

# 1. SETUP
base_folder = "5G 2.6 site list"
result_folder = os.path.join(base_folder, "Result")
os.makedirs(result_folder, exist_ok=True)

def safe_read(path):
    try: return pd.read_csv(path)
    except: return pd.read_csv(path, encoding='latin1')

csv_files = [f for f in glob.glob(os.path.join(base_folder, "*.csv")) if "Result" not in f]
if not csv_files:
    print("No CSV file found!"); exit()

df_main = safe_read(csv_files[0])
df_main.columns = df_main.columns.str.strip()

if 'City' not in df_main.columns:
    print("Error: 'City' column not found in the source CSV!"); exit()

df_main = df_main.dropna(subset=['Lat', 'Lon']).reset_index(drop=True)

# 1.1 DYNAMIC REGEX NORMALIZATION
def dynamic_city_cleaner(city_name):
    if pd.isna(city_name): 
        return "UNKNOWN"
    
    name = str(city_name).upper().strip()
    
    # Global collapse for all Jakarta administrative zones
    if "JAKARTA" in name:
        return "JAKARTA"
        
    # Strip standalone admin markers using word boundaries
    name = re.sub(r'\b(KOTA|KABUPATEN|KAB|ADM)\b', '', name)
    
    # Clean up excess white spaces
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# Establish the seamless boundary grouping
df_main['Cluster_Group'] = df_main['City'].apply(dynamic_city_cleaner)


# 2. HYBRID SEQUENCING LOGIC (For Big Clusters)
def apply_hybrid_sequence(cents, seq_col, is_jabo=False):
    if len(cents) <= 1:
        cents[seq_col] = 1
        return cents
    if is_jabo:
        cents = cents.sort_values('Lon').reset_index(drop=True)
        n_cols = min(8, len(cents))
        cents['col_bin'] = pd.qcut(cents['Lon'], q=n_cols, labels=False, duplicates='drop')
        zigzag_list = []
        for i, col_idx in enumerate(sorted(cents['col_bin'].unique())):
            strip = cents[cents['col_bin'] == col_idx].copy()
            ascending_lat = (i % 2 != 0)
            strip = strip.sort_values('Lat', ascending=ascending_lat)
            zigzag_list.append(strip)
        res = pd.concat(zigzag_list).reset_index(drop=True)
    else:
        cents['start_score'] = cents['Lon'] - cents['Lat']
        start_idx = cents['start_score'].idxmin()
        unvisited = cents.index.tolist()
        current_idx = start_idx
        path = [current_idx]
        unvisited.remove(current_idx)
        coords = cents[['Lat', 'Lon']].values
        while unvisited:
            last_coord = coords[current_idx].reshape(1, -1)
            remaining_coords = coords[unvisited]
            distances = cdist(last_coord, remaining_coords)[0]
            nearest_in_unvisited = unvisited[np.argmin(distances)]
            path.append(nearest_in_unvisited)
            unvisited.remove(nearest_in_unvisited)
            current_idx = nearest_in_unvisited
        res = cents.loc[path].reset_index(drop=True)
    res[seq_col] = range(1, len(res) + 1)
    return res

# 2.1 NANO SEQUENCING (Snake Logic)
def apply_nano_snake_sequence(df_nodes, seq_col):
    if len(df_nodes) <= 1:
        df_nodes[seq_col] = 1
        return df_nodes
    df_nodes = df_nodes.sort_values('Lon').reset_index(drop=True)
    n_cols = max(3, min(6, len(df_nodes) // 4)) 
    df_nodes['col_bin'] = pd.qcut(df_nodes['Lon'], q=n_cols, labels=False, duplicates='drop')
    zigzag_list = []
    for i, col_idx in enumerate(sorted(df_nodes['col_bin'].unique())):
        strip = df_nodes[df_nodes['col_bin'] == col_idx].copy()
        ascending_lat = (i % 2 != 0)
        strip = strip.sort_values('Lat', ascending=ascending_lat)
        zigzag_list.append(strip)
    res = pd.concat(zigzag_list).reset_index(drop=True)
    res[seq_col] = range(1, len(res) + 1)
    return res.drop(columns=['col_bin'])

# 3. CLUSTERING
def strict_balanced_clustering(df, target_size, max_cap, label_prefix):
    n_clusters = max(1, int(np.ceil(len(df) / (target_size - 5))))
    km = KMeans(n_clusters=n_clusters, init='k-means++', n_init=25, random_state=42)
    
    df = df.copy()
    df[label_prefix] = km.fit_predict(df[['Lat', 'Lon']])
    
    final_dfs = []
    current_max = df[label_prefix].max()
    
    for lbl, group in df.groupby(label_prefix):
        group = group.copy()
        if len(group) > max_cap:
            sub_n = int(np.ceil(len(group) / (target_size - 5)))
            sub_km = KMeans(n_clusters=sub_n, init='k-means++', n_init=15, random_state=42)
            group[label_prefix] = sub_km.fit_predict(group[['Lat', 'Lon']]) + (current_max + 1)
            current_max = group[label_prefix].max()
        
        final_dfs.append(group)
        
    return pd.concat(final_dfs).reset_index(drop=True)

def process_hierarchy(df_group, group_name):
    is_jabo_group = group_name.strip().upper() in ["JABO", "JAKARTA", "JABODETABEK"]
    
    df_group = strict_balanced_clustering(df_group, 185, 200, 'big_label')
    bcents = df_group.groupby('big_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
    bcents = apply_hybrid_sequence(bcents, 'big_seq', is_jabo=is_jabo_group)
    
    df_group = df_group.merge(bcents[['big_label', 'big_seq']], on='big_label')
    df_group['big_cluster_id'] = df_group['big_seq'].apply(lambda x: f"{group_name}_{x:02d}")
    
    final_dfs, all_ncents = [], []
    for bid, group in df_group.groupby('big_cluster_id'):
        group = strict_balanced_clustering(group, 25, 30, 'nano_label')
        ncents = group.groupby('nano_label').agg({'Lat':'mean', 'Lon':'mean'}).reset_index()
        ncents = apply_nano_snake_sequence(ncents, 'rel_nano_seq')
        
        group = group.merge(ncents[['nano_label', 'rel_nano_seq']], on='nano_label')
        group['nano_cluster_id'] = group.apply(lambda x: f"{bid}_N{int(x['rel_nano_seq']):02d}", axis=1)
        
        ncents_final = ncents.merge(group[['nano_label', 'nano_cluster_id']].drop_duplicates(), on='nano_label')
        all_ncents.append(ncents_final.sort_values('rel_nano_seq'))
        final_dfs.append(group.drop(columns=['big_label', 'nano_label']))
        
    return pd.concat(final_dfs), bcents.sort_values('big_seq'), all_ncents

# 4. EXECUTION
all_data, b_paths_all, n_paths_all = [], [], []
for cluster_group, group in df_main.groupby('Cluster_Group'):
    df_res, b_p, n_ps = process_hierarchy(group, cluster_group)
    all_data.append(df_res)
    b_paths_all.append(b_p)
    n_paths_all.extend(n_ps)
df_final = pd.concat(all_data).reset_index(drop=True)

# 5. DATA EXPORTS
df_final.to_csv(os.path.join(result_folder, "5G_Master_Site_List.csv"), index=False)

summary_nano = df_final.groupby(['City', 'Cluster_Group', 'big_cluster_id', 'nano_cluster_id']).size().reset_index(name='Total_Sites')
with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Analysis_Report.xlsx")) as writer:
    df_final.to_excel(writer, sheet_name='Site_Details', index=False)
    summary_nano.to_excel(writer, sheet_name='Nano_Cluster_Summary', index=False)

# 6. GEOSPATIAL & VORONOI EXPORT
geometry = [Point(xy) for xy in zip(df_final['Lon'], df_final['Lat'])]
gdf_sites = gpd.GeoDataFrame(df_final, geometry=geometry, crs="EPSG:4326")

def generate_voronoi_shp(gdf, id_col, output_name):
    all_pts = gdf.geometry.unary_union
    bound = all_pts.convex_hull.buffer(0.01)
    vor = voronoi_diagram(all_pts, envelope=bound.envelope)
    vor_gdf = gpd.GeoDataFrame(geometry=list(vor.geoms), crs="EPSG:4326")
    
    # Keeping geometry column explicitly ensures gdf_subset stays a GeoDataFrame for sjoin
    cols_to_keep = ['big_cluster_id', 'nano_cluster_id', 'big_seq', 'rel_nano_seq', 'geometry']
    cols_to_keep = list(set([c for c in cols_to_keep if c in gdf.columns] + [id_col]))
    gdf_subset = gdf[cols_to_keep]
    
    joined = gpd.sjoin(vor_gdf, gdf_subset, how="inner", predicate="contains")
    dissolved = joined.dissolve(by=id_col).reset_index()
    
    # Clip topology boundaries and drop microscopic point/line coordinate artifact layers
    dissolved['geometry'] = dissolved.geometry.intersection(bound)
    dissolved = dissolved[dissolved.geometry.type.isin(['Polygon', 'MultiPolygon'])].reset_index(drop=True)
    
    dissolved.to_file(os.path.join(result_folder, output_name))
    return dissolved

print("Exporting Voronoi Shapefiles...")
big_borders = generate_voronoi_shp(gdf_sites, 'big_cluster_id', "Big_Cluster_Borders.shp")
nano_borders = generate_voronoi_shp(gdf_sites, 'nano_cluster_id', "Nano_Cluster_Borders.shp")

# 7. MAPPING
m = folium.Map(location=[df_final['Lat'].mean(), df_final['Lon'].mean()], zoom_start=11, tiles='CartoDB Positron')
lyr_big_vor = folium.FeatureGroup(name="Big Cluster Borders", show=True)
lyr_sites = folium.FeatureGroup(name="Clusterized Sites", show=True)
lyr_big_path = folium.FeatureGroup(name="Big Cluster Sequence", show=True)
lyr_nano_path = folium.FeatureGroup(name="Nano Mobility Paths (Off)", show=False)

colors = list(mcolors.TABLEAU_COLORS.values()) * 50
big_color_map = {bid: colors[i % len(colors)] for i, bid in enumerate(df_final['big_cluster_id'].unique())}

for _, row in big_borders.iterrows():
    c = big_color_map.get(row['big_cluster_id'], '#808080')
    folium.GeoJson(row['geometry'], style_function=lambda x, c=c: {'fillColor': c, 'color': 'white', 'weight': 2, 'fillOpacity': 0.25}).add_to(lyr_big_vor)

for _, r in df_final.iterrows():
    c = big_color_map[r['big_cluster_id']]
    s_id = r.get('New Sites ID', r.get('Unique ID', 'N/A'))
    popup_html = f"<b>ID:</b> {s_id}<br><b>Original City:</b> {r['City']}<br><b>Big:</b> {r['big_cluster_id']}<br><b>Nano:</b> {r['nano_cluster_id']}"
    folium.CircleMarker([r['Lat'], r['Lon']], radius=3.5, color=c, fill=True, fill_opacity=0.8, weight=1, popup=folium.Popup(popup_html, max_width=200)).add_to(lyr_sites)

def draw_styled_path(df, seq, layer, weight, color, add_labels=False):
    pts = df[['Lat', 'Lon']].values.tolist()
    if len(pts) > 1:
        folium.PolyLine(pts, color='white', weight=weight+2, opacity=0.7).add_to(layer)
        folium.PolyLine(pts, color=color, weight=weight, opacity=1.0).add_to(layer)
    if add_labels:
        for _, p in df.iterrows():
            folium.Marker([p['Lat'], p['Lon']], icon=folium.DivIcon(html=f'<div style="font-size: 7pt; color: white; background: {color}; border-radius: 50%; width: 18px; height: 18px; text-align: center; line-height: 18px; border: 1px solid white; font-weight: bold;">{int(p[seq])}</div>')).add_to(layer)

for b_p in b_paths_all: draw_styled_path(b_p, 'big_seq', lyr_big_path, 4.5, "#1C1C1C", add_labels=True)
for n_p in n_paths_all: draw_styled_path(n_p, 'rel_nano_seq', lyr_nano_path, 3, "#444444")

lyr_big_vor.add_to(m); lyr_sites.add_to(m); lyr_big_path.add_to(m); lyr_nano_path.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(os.path.join(result_folder, "5G_Cluster_Map_Styled.html"))

print("Excel Report, Shapefiles, and Map generated successfully using dynamic administrative grouping.")

C:\Users\m00811323\AppData\Local\Temp\ipykernel_33992\1776695950.py:172: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.3' currently installed).
  with pd.ExcelWriter(os.path.join(result_folder, "5G_Cluster_Analysis_Report.xlsx")) as writer:


Exporting Voronoi Shapefiles...


C:\Users\m00811323\AppData\Local\Temp\ipykernel_33992\1776695950.py:181: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  all_pts = gdf.geometry.unary_union
C:\Users\m00811323\AppData\Local\Temp\ipykernel_33992\1776695950.py:198: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  dissolved.to_file(os.path.join(result_folder, output_name))
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'big_cluster_id' to 'big_cluste'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'index_right' to 'index_righ'
  ogr_write(
c:\users\m00811323\appdata\local\programs\python\python39\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'rel_nano_seq' to 'rel_nano_s'

Excel Report, Shapefiles, and Map generated successfully using dynamic administrative grouping.
